In [4]:
import os

# Set your new working directory
os.chdir('/home/nicholasw/sync/UMD files/research projects/gradient_estimators_for_measles_vaccine/code/WD_LR_grad_est_for_SIR_model')

from pathlib import Path
import ipywidgets as widgets
import matplotlib.pyplot as plt
import pandas as pd

# Load dataset outputs
data_dir = Path("output/data/rev_table1")

df_fd = pd.read_csv(data_dir / "fd_estimators_results.csv")
df_wd_v = pd.read_csv(data_dir / "wd_v_estimators_results.csv")
df_wd_beta = pd.read_csv(data_dir / "wd_beta_estimators_results.csv")
df_lr_v = pd.read_csv(data_dir / "lr_v_estimators_results.csv")
df_lr_beta = pd.read_csv(data_dir / "lr_beta_estimators_results.csv")

# Define metric mappings
METRICS = {
    "mse_x_time": "MSE × Time (Effort-Adjusted Error)",
    "mse": "Mean Squared Error (MSE)",
    "time": "Execution Time (s)",
    "sd": "Standard Deviation (SD)",
}

# Metric & Parameter Controls
metric_dropdown = widgets.Dropdown(
    options=[(label, key) for key, label in METRICS.items()],
    value="mse_x_time",
    description="Metric:",
)

param_radio = widgets.RadioButtons(
    options=[(r"$\beta$", "beta"), ("v", "v")],
    value="beta",
    description="Target Param:",
)

N_slider = widgets.SelectionSlider(
    options=sorted(df_fd["N"].unique()), description="N:"
)
v_slider = widgets.SelectionSlider(
    options=sorted(df_fd["v"].unique()), description="v:"
)
beta_slider = widgets.SelectionSlider(
    options=sorted(df_fd["beta"].unique()), description="beta:"
)
T_slider = widgets.SelectionSlider(
    options=sorted(df_fd["T"].unique()), description="T:"
)
i0_dropdown = widgets.Dropdown(
    options=sorted(df_fd["i0"].unique()), description="i0:"
)
log_y_check = widgets.Checkbox(value=True, description="Log Scale Y-Axis")


def compute_custom_metric(df, param, metric_type):
    """Calculates selected evaluation metrics across dataframe rows."""
    mse_col = f"mse_{param}"
    time_col = f"time_{param}"
    sd_col = f"sd_{param}"

    if metric_type == "mse_x_time":
        return df[mse_col] * df[time_col]
    elif metric_type == "mse":
        return df[mse_col]
    elif metric_type == "time":
        return df[time_col]
    elif metric_type == "sd":
        return df[sd_col]
    else:
        raise ValueError(f"Unknown metric type: {metric_type}")


def plot_metric(metric_type, param, N, v, beta, T, i0, log_y):
    fig, ax = plt.subplots(figsize=(9, 5))

    query = f"N == {N} and v == {v} and beta == {beta} and T == {T} and i0 == {i0}"

    # Filter baseline datasets by slice parameters
    fd_sub = df_fd.query(query).sort_values("eps")
    wd_sub = (
        df_wd_beta.query(query) if param == "beta" else df_wd_v.query(query)
    )
    lr_sub = (
        df_lr_beta.query(query) if param == "beta" else df_lr_v.query(query)
    )

    metric_label = METRICS[metric_type]

    # Plot Finite Difference curve
    if not fd_sub.empty:
        fd_y = compute_custom_metric(fd_sub, param, metric_type)
        ax.plot(
            fd_sub["eps"],
            fd_y,
            "o-",
            color="tab:blue",
            linewidth=2,
            label="Finite Difference (FD)",
        )

    # Plot baseline reference lines for WD and LR
    if not wd_sub.empty:
        wd_val = compute_custom_metric(wd_sub, param, metric_type).iloc[0]
        ax.axhline(
            wd_val,
            color="tab:orange",
            linestyle="--",
            linewidth=2,
            label=f"Weak Derivative (WD) [{wd_val:.2e}]",
        )

    if not lr_sub.empty:
        lr_val = compute_custom_metric(lr_sub, param, metric_type).iloc[0]
        ax.axhline(
            lr_val,
            color="tab:green",
            linestyle=":",
            linewidth=2,
            label=f"Likelihood Ratio (LR) [{lr_val:.2e}]",
        )

    # Labeling
    ax.set_xlabel(r"FD Epsilon ($\epsilon$)", fontsize=11)
    ax.set_ylabel(f"{metric_label}", fontsize=11)
    ax.set_title(
        f"Estimator Performance: {metric_label} ({param})\n"
        f"$N={N},\\ v={v},\\ \\beta={beta},\\ T={T},\\ i_0={i0}$",
        fontsize=12,
    )

    if log_y:
        ax.set_yscale("log")

    ax.grid(True, which="both", linestyle="--", alpha=0.5)
    ax.legend(frameon=True, facecolor="white", framealpha=0.9)
    plt.tight_layout()
    plt.show()


# Display Interactive UI
ui = widgets.VBox(
    [
        widgets.HBox([metric_dropdown, param_radio, log_y_check]),
        widgets.HBox([N_slider, v_slider, beta_slider]),
        widgets.HBox([T_slider, i0_dropdown]),
    ]
)

out = widgets.interactive_output(
    plot_metric,
    {
        "metric_type": metric_dropdown,
        "param": param_radio,
        "N": N_slider,
        "v": v_slider,
        "beta": beta_slider,
        "T": T_slider,
        "i0": i0_dropdown,
        "log_y": log_y_check,
    },
)

display(ui, out)

Output()

In [5]:
from pathlib import Path
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Load dataset outputs
data_dir = Path("output/data/rev_table1")

df_fd = pd.read_csv(data_dir / "fd_estimators_results.csv")
df_wd_v = pd.read_csv(data_dir / "wd_v_estimators_results.csv")
df_wd_beta = pd.read_csv(data_dir / "wd_beta_estimators_results.csv")
df_lr_v = pd.read_csv(data_dir / "lr_v_estimators_results.csv")
df_lr_beta = pd.read_csv(data_dir / "lr_beta_estimators_results.csv")

# Define metric & axis option mappings
METRICS = {
    "mse_x_time": "MSE × Time (Effort-Adjusted Error)",
    "mse": "Mean Squared Error (MSE)",
    "time": "Execution Time (s)",
    "sd": "Standard Deviation (SD)",
}

X_AXIS_OPTIONS = [
    ("N (Population Size)", "N"),
    ("T (Time Horizon)", "T"),
    (r"$\beta$ (Transmission Rate)", "beta"),
    ("v (Vaccination Rate)", "v"),
]

# Interactive Widget Controls
x_axis_dropdown = widgets.Dropdown(
    options=X_AXIS_OPTIONS, value="N", description="X-Axis:"
)

metric_dropdown = widgets.Dropdown(
    options=[(label, key) for key, label in METRICS.items()],
    value="mse_x_time",
    description="Metric:",
)

param_radio = widgets.RadioButtons(
    options=[(r"$\beta$", "beta"), ("v", "v")],
    value="beta",
    description="Target Param:",
)

eps_slider = widgets.SelectionSlider(
    options=sorted(df_fd["eps"].unique()), description="FD Epsilon:"
)

N_slider = widgets.SelectionSlider(
    options=sorted(df_fd["N"].unique()), description="N:"
)
v_slider = widgets.SelectionSlider(
    options=sorted(df_fd["v"].unique()), description="v:"
)
beta_slider = widgets.SelectionSlider(
    options=sorted(df_fd["beta"].unique()), description="beta:"
)
T_slider = widgets.SelectionSlider(
    options=sorted(df_fd["T"].unique()), description="T:"
)
i0_dropdown = widgets.Dropdown(
    options=sorted(df_fd["i0"].unique()), description="i0:"
)
log_y_check = widgets.Checkbox(value=True, description="Log Scale Y-Axis")


def compute_custom_metric(df, param, metric_type):
    """Calculates selected evaluation metrics across dataframe rows."""
    mse_col = f"mse_{param}"
    time_col = f"time_{param}"
    sd_col = f"sd_{param}"

    if metric_type == "mse_x_time":
        return df[mse_col] * df[time_col]
    elif metric_type == "mse":
        return df[mse_col]
    elif metric_type == "time":
        return df[time_col]
    elif metric_type == "sd":
        return df[sd_col]
    else:
        raise ValueError(f"Unknown metric type: {metric_type}")


def get_filtered_subsets(x_axis, param, eps, N, v, beta, T, i0):
    """Filters dataframes along fixed parameters while letting x_axis vary."""
    params = {"N": N, "v": v, "beta": beta, "T": T, "i0": i0}

    wd_df = df_wd_beta if param == "beta" else df_wd_v
    lr_df = df_lr_beta if param == "beta" else df_lr_v

    fd_mask = pd.Series(True, index=df_fd.index)
    wd_mask = pd.Series(True, index=wd_df.index)
    lr_mask = pd.Series(True, index=lr_df.index)

    # Filter out fixed parameter dimensions (excluding current X-axis variable)
    for k, val in params.items():
        if k != x_axis:
            fd_mask &= df_fd[k] == val
            wd_mask &= wd_df[k] == val
            lr_mask &= lr_df[k] == val

    # Match epsilon for Finite Difference
    fd_mask &= (df_fd["eps"] - eps).abs() < 1e-6

    return (
        df_fd[fd_mask].sort_values(x_axis),
        wd_df[wd_mask].sort_values(x_axis),
        lr_df[lr_mask].sort_values(x_axis),
    )


def plot_metric_vs_param(x_axis, metric_type, param, eps, N, v, beta, T, i0, log_y):
    fig, ax = plt.subplots(figsize=(9, 5))

    fd_sub, wd_sub, lr_sub = get_filtered_subsets(
        x_axis, param, eps, N, v, beta, T, i0
    )
    metric_label = METRICS[metric_type]

    # Plot estimator curves vs selected X-axis parameter
    if not fd_sub.empty:
        fd_y = compute_custom_metric(fd_sub, param, metric_type)
        ax.plot(
            fd_sub[x_axis],
            fd_y,
            "o-",
            color="tab:blue",
            linewidth=2,
            label=f"Finite Difference (FD, $\epsilon={eps}$)",
        )

    if not wd_sub.empty:
        wd_y = compute_custom_metric(wd_sub, param, metric_type)
        ax.plot(
            wd_sub[x_axis],
            wd_y,
            "s--",
            color="tab:orange",
            linewidth=2,
            label="Weak Derivative (WD)",
        )

    if not lr_sub.empty:
        lr_y = compute_custom_metric(lr_sub, param, metric_type)
        ax.plot(
            lr_sub[x_axis],
            lr_y,
            "^:",
            color="tab:green",
            linewidth=2,
            label="Likelihood Ratio (LR)",
        )

    # Construct subtitle summarizing remaining fixed slice values
    fixed_params = [
        ("N", N),
        ("v", v),
        (r"\beta", beta),
        ("T", T),
        ("i_0", i0),
    ]
    x_axis_sym = r"\beta" if x_axis == "beta" else x_axis
    fixed_str = ", ".join(
        [
            f"${k}={val}$"
            for k, val in fixed_params
            if k != x_axis and k != x_axis_sym
        ]
    )

    ax.set_xlabel(f"Parameter: {x_axis}", fontsize=11)
    ax.set_ylabel(f"{metric_label}", fontsize=11)
    ax.set_title(
        f"Estimator Performance vs {x_axis}: {metric_label} ({param})\n"
        f"Fixed Parameters: {fixed_str}",
        fontsize=12,
    )

    if log_y:
        ax.set_yscale("log")

    ax.grid(True, which="both", linestyle="--", alpha=0.5)
    ax.legend(frameon=True, facecolor="white", framealpha=0.9)
    plt.tight_layout()
    plt.show()


# Display Interactive UI
ui = widgets.VBox(
    [
        widgets.HBox([x_axis_dropdown, metric_dropdown, param_radio, log_y_check]),
        widgets.HBox([eps_slider, N_slider, v_slider]),
        widgets.HBox([beta_slider, T_slider, i0_dropdown]),
    ]
)

out = widgets.interactive_output(
    plot_metric_vs_param,
    {
        "x_axis": x_axis_dropdown,
        "metric_type": metric_dropdown,
        "param": param_radio,
        "eps": eps_slider,
        "N": N_slider,
        "v": v_slider,
        "beta": beta_slider,
        "T": T_slider,
        "i0": i0_dropdown,
        "log_y": log_y_check,
    },
)

display(ui, out)

Output()

In [6]:
from pathlib import Path
import ipywidgets as widgets
import matplotlib.pyplot as plt
import pandas as pd

# Load dataset outputs
data_dir = Path("output/data/rev_table1")

df_matrix = pd.read_csv(data_dir / "matrix_power_gradients.csv")
df_fd = pd.read_csv(data_dir / "fd_estimators_results.csv")
df_wd_v = pd.read_csv(data_dir / "wd_v_estimators_results.csv")
df_wd_beta = pd.read_csv(data_dir / "wd_beta_estimators_results.csv")
df_lr_v = pd.read_csv(data_dir / "lr_v_estimators_results.csv")
df_lr_beta = pd.read_csv(data_dir / "lr_beta_estimators_results.csv")

# Clean numeric matrix target columns if needed
for col in ["J_v", "J_beta"]:
    if col in df_matrix.columns:
        df_matrix[col] = pd.to_numeric(
            df_matrix[col].astype(str).str.replace('"', ""), errors="coerce"
        )

# Axis & Interactive Controls
X_AXIS_OPTIONS = [
    ("N (Population Size)", "N"),
    ("T (Time Horizon)", "T"),
    (r"$\beta$ (Transmission Rate)", "beta"),
    ("v (Vaccination Rate)", "v"),
]

x_axis_dropdown = widgets.Dropdown(
    options=X_AXIS_OPTIONS, value="N", description="X-Axis:"
)

param_radio = widgets.RadioButtons(
    options=[(r"$\beta$", "beta"), ("v", "v")],
    value="beta",
    description="Target Param:",
)

eps_slider = widgets.SelectionSlider(
    options=sorted(df_fd["eps"].unique()), description="FD Epsilon:"
)

N_slider = widgets.SelectionSlider(
    options=sorted(df_fd["N"].unique()), description="N:"
)
v_slider = widgets.SelectionSlider(
    options=sorted(df_fd["v"].unique()), description="v:"
)
beta_slider = widgets.SelectionSlider(
    options=sorted(df_fd["beta"].unique()), description="beta:"
)
T_slider = widgets.SelectionSlider(
    options=sorted(df_fd["T"].unique()), description="T:"
)
i0_dropdown = widgets.Dropdown(
    options=sorted(df_fd["i0"].unique()), description="i0:"
)

# Series Visibility Checkboxes
show_matrix_check = widgets.Checkbox(
    value=True, description="Matrix Target (True)", indent=False
)
show_fd_check = widgets.Checkbox(
    value=True, description="FD Estimator", indent=False
)
show_wd_check = widgets.Checkbox(
    value=True, description="WD Estimator", indent=False
)
show_lr_check = widgets.Checkbox(
    value=True, description="LR Estimator", indent=False
)


def get_ci_subsets(x_axis, param, eps, N, v, beta, T, i0):
    """Filters all datasets to vary along x_axis while keeping other parameters fixed."""
    params = {"N": N, "v": v, "beta": beta, "T": T, "i0": i0}

    wd_df = df_wd_beta if param == "beta" else df_wd_v
    lr_df = df_lr_beta if param == "beta" else df_lr_v

    mat_mask = pd.Series(True, index=df_matrix.index)
    fd_mask = pd.Series(True, index=df_fd.index)
    wd_mask = pd.Series(True, index=wd_df.index)
    lr_mask = pd.Series(True, index=lr_df.index)

    for k, val in params.items():
        if k != x_axis:
            mat_mask &= df_matrix[k] == val
            fd_mask &= df_fd[k] == val
            wd_mask &= wd_df[k] == val
            lr_mask &= lr_df[k] == val

    fd_mask &= (df_fd["eps"] - eps).abs() < 1e-6

    return (
        df_matrix[mat_mask].sort_values(x_axis),
        df_fd[fd_mask].sort_values(x_axis),
        wd_df[wd_mask].sort_values(x_axis),
        lr_df[lr_mask].sort_values(x_axis),
    )


def plot_gradient_ci(
    x_axis,
    param,
    eps,
    N,
    v,
    beta,
    T,
    i0,
    show_matrix,
    show_fd,
    show_wd,
    show_lr,
):
    fig, ax = plt.subplots(figsize=(9, 5))

    mat_sub, fd_sub, wd_sub, lr_sub = get_ci_subsets(
        x_axis, param, eps, N, v, beta, T, i0
    )

    mean_col = f"mean_{param}"
    sd_col = f"sd_{param}"
    target_col = f"J_{param}"

    # 1. Ground Truth Target (Matrix Power)
    if show_matrix and not mat_sub.empty:
        ax.plot(
            mat_sub[x_axis],
            mat_sub[target_col],
            "k*--",
            linewidth=1.8,
            markersize=9,
            label="Matrix Power Target (True Gradient)",
        )

    # 2. Finite Difference (FD)
    if show_fd and not fd_sub.empty:
        x = fd_sub[x_axis]
        mean = fd_sub[mean_col]
        sd = fd_sub[sd_col]
        ax.plot(
            x,
            mean,
            "o-",
            color="tab:blue",
            linewidth=2,
            label=f"Finite Difference (FD, $\epsilon={eps}$)",
        )
        ax.fill_between(
            x, mean - 2 * sd, mean + 2 * sd, color="tab:blue", alpha=0.2
        )

    # 3. Weak Derivative (WD)
    if show_wd and not wd_sub.empty:
        x = wd_sub[x_axis]
        mean = wd_sub[mean_col]
        sd = wd_sub[sd_col]
        ax.plot(
            x,
            mean,
            "s--",
            color="tab:orange",
            linewidth=2,
            label="Weak Derivative (WD)",
        )
        ax.fill_between(
            x, mean - 2 * sd, mean + 2 * sd, color="tab:orange", alpha=0.2
        )

    # 4. Likelihood Ratio (LR)
    if show_lr and not lr_sub.empty:
        x = lr_sub[x_axis]
        mean = lr_sub[mean_col]
        sd = lr_sub[sd_col]
        ax.plot(
            x,
            mean,
            "^:",
            color="tab:green",
            linewidth=2,
            label="Likelihood Ratio (LR)",
        )
        ax.fill_between(
            x, mean - 2 * sd, mean + 2 * sd, color="tab:green", alpha=0.2
        )

    # Formatting
    fixed_params = [
        ("N", N),
        ("v", v),
        (r"\beta", beta),
        ("T", T),
        ("i_0", i0),
    ]
    x_axis_sym = r"\beta" if x_axis == "beta" else x_axis
    fixed_str = ", ".join(
        [
            f"${k}={val}$"
            for k, val in fixed_params
            if k != x_axis and k != x_axis_sym
        ]
    )

    grad_str = (
        r"\partial J / \partial \beta"
        if param == "beta"
        else r"\partial J / \partial v"
    )
    ax.set_xlabel(f"Parameter: {x_axis}", fontsize=11)
    ax.set_ylabel(f"Gradient Estimate (${grad_str}$)", fontsize=11)
    ax.set_title(
        f"Gradient Estimators vs Ground Truth ($2\sigma$ CI) [{param}]\n"
        f"Fixed Parameters: {fixed_str}",
        fontsize=12,
    )

    ax.grid(True, which="both", linestyle="--", alpha=0.5)

    # Only show legend if at least one series is visible
    if show_matrix or show_fd or show_wd or show_lr:
        ax.legend(frameon=True, facecolor="white", framealpha=0.9)

    plt.tight_layout()
    plt.show()


# Display Interactive UI
ui = widgets.VBox(
    [
        widgets.HBox([x_axis_dropdown, param_radio]),
        widgets.HBox(
            [
                show_matrix_check,
                show_fd_check,
                show_wd_check,
                show_lr_check,
            ]
        ),
        widgets.HBox([eps_slider, N_slider, v_slider]),
        widgets.HBox([beta_slider, T_slider, i0_dropdown]),
    ]
)

out = widgets.interactive_output(
    plot_gradient_ci,
    {
        "x_axis": x_axis_dropdown,
        "param": param_radio,
        "eps": eps_slider,
        "N": N_slider,
        "v": v_slider,
        "beta": beta_slider,
        "T": T_slider,
        "i0": i0_dropdown,
        "show_matrix": show_matrix_check,
        "show_fd": show_fd_check,
        "show_wd": show_wd_check,
        "show_lr": show_lr_check,
    },
)

display(ui, out)

Output()